# Notebook 2 — MAP denoising with an MRF prior

We now denoise for real by minimising the MAP objective
\[
\hat{x}=\argmin_x\; \tfrac12\norm{x-y}_2^2 \;+\; \lambda \sum_{i\sim j} g(x_i-x_j),
\]
where the sum is over neighbouring pixel pairs (a 4-neighbour MRF) and \(g\) is a
clique penalty. We compare two priors and two optimisers:

- **Quadratic** penalty \(g(u)=\tfrac12 u^2\) (a Gaussian MRF). Its "force"
  \(g'(u)=u\) keeps growing, so it fights every jump and **blurs edges**.
- **Huber** penalty (quadratic near 0, linear far out). Its force **saturates**,
  so big jumps (real edges) are left alone — **edge-preserving**.

Optimisers: an **ICM-style** local-mode update (closed form for the quadratic
prior) and **gradient descent** (works for any penalty). NumPy + Matplotlib only.

\(\lambda\) is the smoothness weight (it plays the role of \(\beta\sigma^2\) from
the slides): too small leaves noise, too large over-smooths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
plt.rcParams['figure.figsize'] = (10, 4)

def phantom(N=128):
    yy, xx = np.mgrid[0:N, 0:N]
    img = np.full((N, N), 0.2)
    img[(xx - 64)**2 + (yy - 64)**2 < 34**2] = 0.8   # disk: a big, sharp edge
    img[20:45, 20:108] = 0.5                          # a bar: a smaller edge
    return img

def psnr(ref, test, peak=1.0):
    mse = np.mean((ref - test)**2)
    return 10*np.log10(peak**2 / mse)

clean = phantom()
sigma = 0.12
noisy = clean + rng.normal(0.0, sigma, clean.shape)

print(f'noisy PSNR = {psnr(clean, noisy):.2f} dB')
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(clean, cmap='gray', vmin=0, vmax=1); ax[0].set_title('clean'); ax[0].axis('off')
ax[1].imshow(noisy, cmap='gray', vmin=0, vmax=1); ax[1].set_title(f'noisy (sigma={sigma})'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## The prior gradient and the two penalties

The gradient of the prior energy at pixel \(i\) is \(\sum_{j\in N_i} g'(x_i-x_j)\).
We only need the **derivative** \(\psi=g'\) of the penalty:

- quadratic: \(\psi(u)=u\) — linear, **never saturates** (force grows with the jump);
- Huber: \(\psi(u)=\mathrm{clip}(u,-\delta,\delta)\) — **saturates** at \(\pm\delta\),
  so deviations bigger than \(\delta\) (real edges) feel a constant, bounded force.

`prior_gradient` sums \(\psi(x_i-x_j)\) over the 4 neighbours using edge
replication (so border pixels feel no spurious force).

In [ ]:
def prior_gradient(x, psi):
    """Sum over the 4 neighbours of psi(x_i - x_j), edge-replicated."""
    xp = np.pad(x, 1, mode='edge')
    up,   down  = xp[:-2, 1:-1], xp[2:,  1:-1]
    left, right = xp[1:-1, :-2], xp[1:-1, 2:]
    return psi(x - up) + psi(x - down) + psi(x - left) + psi(x - right)

def psi_quadratic(u):
    return u                                    # force grows forever -> blurs edges

def psi_huber(u, delta=0.25):
    return np.clip(u, -delta, delta)            # force saturates -> preserves edges

def map_denoise_gd(y, psi, lam, iters=600, step=None):
    """Gradient descent on 0.5||x-y||^2 + lam * sum_pairs g(diff)."""
    if step is None:
        step = 1.0 / (1.0 + 8.0*lam)            # ~1/Lipschitz (psi' <= 1): always stable
    x = y.copy()
    for _ in range(iters):
        grad = (x - y) + lam * prior_gradient(x, psi)
        x = x - step * grad
    return x

print('helpers ready')

## Optimiser 1 — ICM-style local update (quadratic prior)

For a Gaussian data term and a **quadratic** prior, the local conditional
posterior at pixel \(i\) is Gaussian, and its mode has a closed form:
\[
x_i \leftarrow \frac{y_i + \lambda \sum_{j\in N_i} x_j}{1 + \lambda\,|N_i|}.
\]
Sweeping this update is exactly ICM's "move each pixel to its local mode". Here
we apply it to all pixels at once (a Jacobi/parallel sweep, as the slides note
for practical speed).

In [ ]:
def icm_quadratic(y, lam, passes=80):
    x = y.copy()
    for _ in range(passes):
        xp = np.pad(x, 1, mode='edge')
        neigh = xp[:-2, 1:-1] + xp[2:, 1:-1] + xp[1:-1, :-2] + xp[1:-1, 2:]
        x = (y + lam*neigh) / (1.0 + 4.0*lam)
    return x

icm = icm_quadratic(noisy, lam=1.0, passes=80)
print(f'ICM (quadratic) PSNR = {psnr(clean, icm):.2f} dB   (noisy was {psnr(clean, noisy):.2f} dB)')

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a_, im, t in zip(ax, [clean, noisy, icm], ['clean', 'noisy', 'ICM (quadratic)']):
    a_.imshow(im, cmap='gray', vmin=0, vmax=1); a_.set_title(t); a_.axis('off')
plt.tight_layout(); plt.show()

## Optimiser 2 — gradient descent: quadratic vs Huber (the edge test)

Now the headline experiment. Denoise with gradient descent using each penalty,
then look at a **line profile across the disk edge**. The quadratic prior rounds
the step off (blur); the Huber prior keeps it vertical (edge preserved).

In [ ]:
den_quad  = map_denoise_gd(noisy, psi_quadratic, lam=0.5)
den_huber = map_denoise_gd(noisy, psi_huber,     lam=0.8)
print(f'quadratic prior PSNR = {psnr(clean, den_quad):.2f} dB')
print(f'Huber prior     PSNR = {psnr(clean, den_huber):.2f} dB')

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a_, im, t in zip(ax, [noisy, den_quad, den_huber, clean],
                     ['noisy', 'quadratic (blurs)', 'Huber (keeps edges)', 'clean']):
    a_.imshow(im, cmap='gray', vmin=0, vmax=1); a_.set_title(t); a_.axis('off')
plt.tight_layout(); plt.show()

row = 64   # cuts straight through the disk
plt.figure(figsize=(9, 4))
plt.plot(clean[row], 'k-', lw=2, label='clean')
plt.plot(noisy[row], color='0.75', lw=0.8, label='noisy')
plt.plot(den_quad[row], 'b-', label='quadratic')
plt.plot(den_huber[row], 'r-', label='Huber')
plt.xlim(20, 108); plt.xlabel('column'); plt.ylabel('intensity')
plt.legend(); plt.title('profile across the disk edge: Huber stays sharp'); plt.show()

## The smoothness knob λ (β in the slides)

Sweep λ and watch PSNR rise then fall: too little regularisation leaves noise,
too much erases the signal. The peak is the sweet spot you tune in practice.
Note the Huber prior usually peaks higher, because it can smooth hard without
paying for it at the edges.

In [ ]:
lams = np.array([0.05, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2])
ps_quad = [psnr(clean, map_denoise_gd(noisy, psi_quadratic, l)) for l in lams]
ps_hub  = [psnr(clean, map_denoise_gd(noisy, psi_huber,     l)) for l in lams]

plt.figure(figsize=(8, 4))
plt.semilogx(lams, ps_quad, 'o-', label='quadratic prior')
plt.semilogx(lams, ps_hub,  's-', label='Huber prior')
plt.axhline(psnr(clean, noisy), color='gray', ls=':', label='noisy baseline')
plt.xlabel('lambda (smoothness weight)'); plt.ylabel('PSNR (dB)')
plt.legend(); plt.title('too little -> noisy, too much -> over-smoothed'); plt.show()

best = lams[int(np.argmax(ps_hub))]
print(f'best Huber lambda here ~ {best}  ->  PSNR {max(ps_hub):.2f} dB')

## Exercises

1. Add **Rician** noise (from Notebook 1) instead of Gaussian and denoise. Does the Gaussian-data-term denoiser still work? Where does it fail near dark regions?
2. Replace the Huber `delta` with a few values (0.05, 0.15, 0.4). How does it trade noise removal against edge sharpness?
3. Implement a **true sequential ICM** (update pixels one at a time in raster order) and compare its result and speed to the parallel sweep.
4. Add a tiny "salt-and-pepper" outlier and watch the quadratic prior smear it while the Huber prior shrugs it off.
5. Swap in a **total-variation** style penalty \(\psi(u)=\mathrm{sign}(u)\) (with a small smoothing near 0) and compare to Huber.

**Next:** Notebook 3 keeps the same MAP machinery but puts a real operator \(G\) in the middle — deblurring, CT, and MRI reconstruction.